# Exercise: Data Modeling with Apache Iceberg Tables

The fastest query is one that doesn't have to actually read data off disk and all table
formats are designed around this rule. Data modeling in Apache Iceberg is about organizing
our data into files that can be excluded from query processing without actually opening
them up. Every file we can exclude based on partition or metric information is saving
time without sacrificing correctness.

In this exercise, you'll learn how to:
- Work with real-world data (NYC Taxi dataset)
- Create tables with different partitioning strategies and Sort Orders
- Analyze table metadata to understand the impact of partitioning on file layout
- Test different query patterns and pushdown optimizations
- Use Spark UI to understand query execution


## Initialize Spark Session

First, let's start our Spark session. This may take a while if starting for the first time as dependencies are downloaded.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("DataModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

Spark 4.0.1 initialized!


## Create Namespace


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.taxi")
print("Namespace 'taxi' created!")

Namespace 'taxi' created!


## Download NYC Taxi Data

We'll download NYC Yellow Taxi trip data from **June through October 2023** (5 months). This dataset contains:
- Pick-up and drop-off times and locations
- Trip distances
- Fare amounts
- Payment types
- Passenger counts

**Note**: We're downloading 5 months of data (~225MB total). This will give us enough data to see meaningful differences between partitioning strategies.

In [3]:
import urllib.request
import os
import boto3
from botocore.client import Config

# Configure MinIO client using S3-compatible API
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download NYC Yellow Taxi data for June through October 2023 and upload to MinIO
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
minio_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

# Create temp directory
os.makedirs(temp_dir, exist_ok=True)

# Download data for months 6-10 (June through October)
months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    minio_key = f"{minio_prefix}/{filename}"
    
    try:
        # Check if file already exists locally
        if os.path.exists(local_path):
            print(f"{filename} already exists locally, skipping download")
        else:
            # Download file
            print(f"Downloading {filename} (~45MB)...")
            urllib.request.urlretrieve(url, local_path)
            print(f"  Downloaded to {local_path}")
        
        # Upload to MinIO
        print(f"  Uploading to MinIO: s3a://{bucket_name}/{minio_key}...")
        s3_client.upload_file(local_path, bucket_name, minio_key)
        print(f"  Uploaded successfully!")
        uploaded_files.append(minio_key)
        
        # Clean up local file
        os.remove(local_path)
    except Exception as e:
        print(f"  Error processing {filename}: {e}")

print(f"\nAll files ready in MinIO! Total: {len(uploaded_files)} months of data")
print(f"Location: s3a://{bucket_name}/{minio_prefix}/")

  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-06.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-06.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-07.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-07.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-08.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-08.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-09.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-09.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-10.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-10.parquet...
  Uploaded successfully!

All files ready in MinIO! Total: 5 months of data
Location: s3a://warehouse/raw/


## Load and Explore the Data

Let's load the parquet files and examine their schema and sample data.

In [4]:
# Read all parquet files from MinIO
s3_path = "s3a://warehouse/raw/"

taxi_df = spark.read.parquet(s3_path)

print(f"Reading from: {s3_path}")
print(f"Total records: {taxi_df.count():,}")
print(f"\nSchema:")
taxi_df.printSchema()

Reading from: s3a://warehouse/raw/
Total records: 15,407,558

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
# Show sample data
print("Sample records:")
taxi_df.show(5, truncate=False)

Sample records:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2023-06-01 00:08:48 |2023-06-01 00:29:41  |1              |3.4          |1         |N                 |140         |238         |1           |21.9       |3.5  |0

In [6]:
# Get date range and basic statistics
print("Date range:")
taxi_df.select(
    F.min("tpep_pickup_datetime").alias("min_date"),
    F.max("tpep_pickup_datetime").alias("max_date")
).show()

print("\nBasic statistics:")
taxi_df.select(
    F.count("*").alias("total_trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.avg("total_amount"), 2).alias("avg_fare"),
    F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("distinct_days")
).show()

Date range:
+-------------------+-------------------+
|           min_date|           max_date|
+-------------------+-------------------+
|2002-12-31 22:27:05|2023-11-23 13:40:43|
+-------------------+-------------------+


Basic statistics:
+-----------+------------+--------+-------------+
|total_trips|avg_distance|avg_fare|distinct_days|
+-----------+------------+--------+-------------+
|   15407558|        4.35|   29.05|          161|
+-----------+------------+--------+-------------+



## Creating Iceberg Tables with Different Partitioning Specifications

Iceberg table partitioning is done using a series of partition transform functions which we call a "spec". An Iceberg Table can have multiple active specs at the same time but for this exercise we'll use a single spec per table to show the different effects partitioning has on data file size and query performance. We'll use CREATE TABLE as SELECT (CTAS) Statements to copy the existing Taxi data into new files using our specified partitioning. This is an expensive approach to creating Iceberg tables from parquet files and we'll go over other methods in future modules. 

*Note: We are setting the deafult target file size to 16MB for these tables inorder to better illustrate the effects of partitioning and pruning*

## Strategy 1: Unpartitioned Table

First, let's create a baseline table without partitioning

In [7]:
# Create unpartitioned table using CTAS
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_unpartitioned
    USING iceberg
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Unpartitioned table created!")

ConnectionRefusedError: [Errno 111] Connection refused

### Analyze Unpartitioned Table Metadata

Iceberg provides [**metadata tables**](https://iceberg.apache.org/docs/latest/spark-queries/#inspecting-tables) that let you inspect the physical layout without scanning data. 
In every engine these are accessed differently. Spark uses a "catalog.namespace.table.metadata_table" pattern.

In [ ]:
# Check the files metadata table
print("Files in unpartitioned table:")
spark.sql("""
    SELECT 
        file_path,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count,
        readable_metrics
    FROM polaris.taxi.trips_unpartitioned.files
""").show(truncate=False)

In [ ]:
# Check snapshots
print("Snapshots:")
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation,
        summary
    FROM polaris.taxi.trips_unpartitioned.snapshots
""").show(truncate=False)

In [ ]:
print("Partitions (should show 1 null partition):")
spark.sql("""
    SELECT 
        partition,
        SUM(record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_unpartitioned.entries
    GROUP BY partition
""").show()

### Check MinIO for Unpartitioned Table

**MinIO Location**: http://localhost:9001/browser/warehouse/taxi/trips_unpartitioned/

## Strategy 2: Monthly Partitioning

Now let's partition by **month**. Iceberg supports temporal transforms like `month()` that automatically extract the month from a timestamp abd store
the value in the table metadata. When partitioning is set in Iceberg, all new files written to the table will all have the same value for the output of the partition transform. That means in this case, data files we create will only have a single month in each of them. This means queries based on time only have to read data files with the same "month" as required by predicates of the query.

In [ ]:
# Create monthly partitioned table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Monthly partitioned table created!")

### Analyze Monthly Partitioned Table

In [ ]:
# Check files - note the partition column
print("Files in monthly partitioned table:")
spark.sql("""
    SELECT 
        file_path,
        partition,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count
    FROM polaris.taxi.trips_by_month.files
""").show(truncate=False)

In [ ]:
# Check partition statistics
# Note we are using the "entires" table because it parallelizes better on Spark than the "partitions" table
print("Partition statistics:")
spark.sql("""
    SELECT 
        partition,
        SUM(record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_month.entries
    GROUP BY partition
    ORDER BY partition
""").show()

### Check MinIO for Monthly Partitioned Table

**MinIO Location**: http://localhost:9001/browser/warehouse/taxi/trips_by_month/

**Important Note**: The partition directories you see (e.g., `tpep_pickup_datetime_month=2023-01/`) are a convenience for humans browsing the storage. Iceberg's actual partition tracking is metadata-driven - it stores partition values in manifest files, not in directory paths. The directories exist because Iceberg writes data files with these prefixes for human readers but the structure is not used during query planning.

## Strategy 3: Daily Partitioning

We can also use the "days" transform to partition our data into files where every row shares a single value for day. This will end up generating much smaller files (1/30th) the size of our month tables but they will be much more selective. Most engines currently do not perform well with small files, so this is generally too small for good performance, but we will address this in later modules.

In [ ]:
# Create daily partitioned table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Daily partitioned table created!")

### Analyze Daily Partitioned Table

In [ ]:
# Check partition statistics - note we have many more partitions now
print("Partition statistics (showing first 10):")
spark.sql("""
    SELECT 
        partition,
        SUM(record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_day.entries
    GROUP BY partition
    ORDER BY partition
    LIMIT 10
""").show()

In [ ]:
# Count total partitions
partition_count = spark.sql("""
    SELECT COUNT(DISTINCT partition) as partition_count 
    FROM polaris.taxi.trips_by_day.entries
""").collect()[0][0]

print(f"Total partitions in daily partitioned table: {partition_count}")

In [ ]:
# Check file distribution across partitions
print("File count per partition (sample):")
spark.sql("""
    SELECT 
        partition,
        COUNT(*) as file_count,
        SUM(record_count) as total_records,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.taxi.trips_by_day.files
    GROUP BY partition
    ORDER BY partition
    LIMIT 10
""").show()

### Check MinIO for Daily Partitioned Table

**MinIO Location**: http://localhost:9001/browser/warehouse/taxi/trips_by_day/

## Strategy 4: Partition by Categorical Column

Let's partition by **pickup location zone** (PULocationID). This is useful when queries frequently filter by location.

**Use case**: Location-based analytics, regional reports, geo-partitioned data.

In [ ]:
# First, let's see the distribution of pickup locations
print("Top 10 pickup locations by trip count:")
spark.sql("""
    SELECT 
        PULocationID,
        COUNT(*) as trip_count
    FROM parquet.`s3a://warehouse/raw/`
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
""").show()

In [ ]:
# Create location partitioned table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_location
    USING iceberg
    PARTITIONED BY (PULocationID)
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Location partitioned table created!")

### Analyze Location Partitioned Table

In [ ]:
# Check partition statistics
print("Partition statistics (top 10 by record count):")
spark.sql("""
    SELECT 
        partition,
        SUM(record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_location.entries
    GROUP BY partition
    ORDER BY record_count DESC
    LIMIT 10
""").show()

In [ ]:
# Analyze partition skew
print("Partition size distribution:")
spark.sql("""
    SELECT 
        COUNT(DISTINCT partition) as total_partitions,
        MIN(record_count) as min_records,
        MAX(record_count) as max_records,
        ROUND(AVG(record_count), 0) as avg_records,
        ROUND(STDDEV(record_count), 0) as stddev_records
    FROM (
        SELECT 
            partition,
            SUM(record_count) as record_count
        FROM polaris.taxi.trips_by_location.entries
        GROUP BY partition
    )
""").show()

print("\nNote the skew: Some locations have many more trips than others!")

### Check MinIO for Location Partitioned Table

**MinIO Location**: http://localhost:9001/browser/warehouse/taxi/trips_by_location/

**What to observe**:
- Navigate to the `data/` folder
- You'll see directories for each location ID (e.g., `PULocationID=132/`)
- Look at file sizes in popular vs. unpopular locations
- This demonstrates data skew - some partitions are much larger than others

**Important Note**: The partition directories (e.g., `PULocationID=132/`) are created as a side-effect of how Iceberg writes data files with partition values in their paths. These directories make it easy for humans to browse and understand the data layout, but Iceberg's query planning doesn't rely on listing directories. All partition information is stored in and retrieved from Iceberg's metadata files.

## Strategy 5: Monthly Partition with Sort Order

Finally, let's create a table that's partitioned by month AND sorted by pickup location. The sort order helps with:
- **Min/Max filtering**: Skip files based on column statistics
- **Range queries**: Efficiently find data within value ranges
- **Clustering**: Related data is stored together

**Use case**: Queries that filter by both month and location (e.g., "all trips from a specific zone in a given month").

In [ ]:
# Create monthly partitioned table with sort order on pickup location
# Note: We need to create the table first, then insert with sorting
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_sorted
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

# Now set the sort order
spark.sql("""
    ALTER TABLE polaris.taxi.trips_sorted
    WRITE ORDERED BY PULocationID
""")

print("Sorted table created!")
print("  - Partitioned by: months(tpep_pickup_datetime)")
print("  - Sorted by: PULocationID")

### Analyze Sorted Table

The key benefit of sorting is visible in the min/max bounds of each file.

In [ ]:
# Check file metadata with bounds
print("File statistics (showing readable metrics including min/max for PULocationID):")
spark.sql("""
    SELECT 
        file_path,
        partition,
        record_count,
        readable_metrics
    FROM polaris.taxi.trips_sorted.files
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Check the table's sort order configuration
print("Table properties:")
spark.sql("""
    DESCRIBE EXTENDED polaris.taxi.trips_sorted
""").filter("col_name LIKE '%sort%' OR col_name LIKE '%order%'").show(truncate=False)

In [ ]:
import pandas as pd

tables = [
    'trips_unpartitioned',
    'trips_by_month',
    'trips_by_day',
    'trips_by_location',
    'trips_sorted'
]

comparison_data = []

for table in tables:
    # Get file count and partition count
    file_stats = spark.sql(f"""
        SELECT 
            COUNT(*) as file_count,
            SUM(record_count) as total_records,
            ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
        FROM polaris.taxi.{table}.files
    """).collect()[0]
    
    partition_count = spark.sql(f"""
        SELECT COUNT(DISTINCT partition) as partition_count 
        FROM polaris.taxi.{table}.entries
    """).collect()[0][0]
    
    comparison_data.append({
        'Table': table,
        'Partitions': partition_count,
        'Files': file_stats['file_count'],
        'Total Records': f"{file_stats['total_records']:,}",
        'Size (MB)': file_stats['total_size_mb']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nPartitioning Strategy Comparison:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("\n" + "=" * 80)

## Query Testing: Understanding Pushdown Optimizations

Now let's test different query patterns to see how partitioning and metadata affect performance.

### Types of Pushdown:
1. **No Pushdown**: Full table scan, all files read
2. **Partition Pushdown**: Skip entire partitions based on filter
3. **Metric Pushdown**: Skip files within partitions based on min/max statistics

### Query 1: No Predicate Pushdown (Full Scan)

This query has no filters, so all data must be read.

In [ ]:
# Query without any filters - full scan
result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
""")

print("Full scan query results:")
result.show()

print("\nCheck Spark UI (http://localhost:4040) to see:")
print("   - All partitions were scanned")
print("   - All files were read")
print("   - Look at the 'Input Size' metric")

### Query 2: Partition Pushdown

This query filters by date, which allows Iceberg to skip entire partitions.

In [ ]:
# Query with date filter - partition pushdown
result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""")

print("Partition pushdown query results (August 2023 only):")
result.show()

print("\nCheck Spark UI (http://localhost:4040) to see:")
print("   - Only partitions for August were scanned (~31 days)")
print("   - Significantly less data read compared to full scan (5 months = ~153 days)")
print("   - Look at 'Tasks' to see fewer tasks than total partitions")

### Query 3: Metric Pushdown (Min/Max Filtering)

This query filters by a non-partition column with a range predicate. Iceberg can use file-level min/max statistics to skip files.

In [ ]:
# Query with location filter - metric pushdown
result = spark.sql("""
    SELECT 
        COUNT(*) as trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.taxi.trips_sorted
    WHERE PULocationID IN (132, 138, 161, 230, 237)
""")

print("Metric pushdown query results (specific locations):")
result.show()

print("\nCheck Spark UI (http://localhost:4040) to see:")
print("   - Files with location ranges outside our filter were skipped")
print("   - The sorted table helps here: locations are clustered within each month")
print("   - Compare input size to the full table size")

### Query 4: Combined Partition + Metric Pushdown

This query combines both: partition pruning (by month) and metric filtering (by location).

In [ ]:
# Query with both month and location filters
result = spark.sql("""
    SELECT 
        COUNT(*) as matching_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.taxi.trips_sorted
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
      AND PULocationID IN (132, 138, 161, 230, 237)
""")

print("Combined pushdown query results (August 2023, specific locations):")
result.show()

print("\nCheck Spark UI (http://localhost:4040) to see:")
print("   - First, partitions outside August were skipped (~4 months)")
print("   - Then, files with location ranges outside our filter were skipped within August")
print("   - This is the most efficient query pattern")

## Understanding Query Plans with EXPLAIN

Spark's EXPLAIN command shows how it plans to execute a query.

### Explain: Full Scan (No Pushdown)

In [ ]:
# Explain a full scan query
spark.sql("""
    EXPLAIN EXTENDED
    SELECT COUNT(*) 
    FROM polaris.taxi.trips_by_day
""").show(truncate=False)

print("\nKey points to look for:")
print("   - BatchScan: Shows the Iceberg scan operation")
print("   - No filters in the plan")
print("   - All partitions will be read")

### Explain: Partition Pushdown

In [ ]:
# Explain a partition pushdown query
spark.sql("""
    EXPLAIN EXTENDED
    SELECT COUNT(*) 
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""").show(truncate=False)

print("\nKey points to look for:")
print("   - Filter: Shows the date predicate (August 2023)")
print("   - PushedFilters: Shows what was pushed down to Iceberg")
print("   - Partition filters are applied before reading data")

### Explain: Metric Pushdown

In [ ]:
# Explain a metric pushdown query
spark.sql("""
    EXPLAIN EXTENDED
    SELECT COUNT(*) 
    FROM polaris.taxi.trips_sorted
    WHERE PULocationID IN (132, 138, 161)
""").show(truncate=False)

print("\nKey points to look for:")
print("   - Filter on PULocationID")
print("   - Iceberg will use file-level statistics")
print("   - Files are skipped based on min/max values")

## Deep Dive: Spark UI Analysis

Let's run a comparison query and then explore the Spark UI.

**Spark UI**: http://localhost:4040

In [ ]:
# Run a query we can analyze in detail
result = spark.sql("""
    SELECT 
        DATE(tpep_pickup_datetime) as trip_date,
        COUNT(*) as trip_count,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-09-01'
      AND tpep_pickup_datetime < '2023-10-01'
      AND total_amount > 20.0
    GROUP BY DATE(tpep_pickup_datetime)
    ORDER BY trip_date
""")

print("Query results:")
result.show()

print("\n" + "=" * 80)
print("NOW GO TO THE SPARK UI: http://localhost:4040")
print("=" * 80)
print("\nWhat to explore:")
print("\n1. SQL Tab:")
print("   - Find the most recent query")
print("   - Click on the query description")
print("   - Look at 'Duration', 'Input Size', and 'Records Read'")
print("\n2. DAG Visualization:")
print("   - Shows the query execution plan as a graph")
print("   - Each box represents a stage")
print("   - Green = completed successfully")
print("\n3. Details Section:")
print("   - Scan iceberg: Shows partition and file filtering")
print("   - Note 'number of files' vs 'total files in table'")
print("   - This shows how many files were SKIPPED")
print("\n4. Stages Tab:")
print("   - Click on a stage ID")
print("   - See task-level details")
print("   - Each task typically processes one or more files")
print("   - Look at 'Input Size / Records' per task")
print("\n5. Key Metrics to Compare:")
print("   - Input Size: How much data was read from storage")
print("   - Shuffle Read/Write: Data movement between stages")
print("   - Task Time: Time spent processing")
print("   - GC Time: Garbage collection overhead")
print("\n6. Executors Tab:")
print("   - Shows resource usage")
print("   - In local mode, you'll see one executor")
print("   - In cluster mode, you'd see multiple executors")

## Performance Comparison: Partitioned vs Unpartitioned

Let's run the same query against both the partitioned and unpartitioned tables to see the difference.

In [ ]:
import time

# Query on unpartitioned table (filtering for August 2023)
start = time.time()
result_unpat = spark.sql("""
    SELECT COUNT(*) as count
    FROM polaris.taxi.trips_unpartitioned
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""").collect()
time_unpat = time.time() - start

print(f"Unpartitioned table (filtering August from 5 months):")
print(f"  Result: {result_unpat[0][0]:,} trips")
print(f"  Time: {time_unpat:.3f} seconds")

In [ ]:
# Query on partitioned table (only scans August partitions)
start = time.time()
result_part = spark.sql("""
    SELECT COUNT(*) as count
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""").collect()
time_part = time.time() - start

print(f"Partitioned table (only scans August partitions):")
print(f"  Result: {result_part[0][0]:,} trips")
print(f"  Time: {time_part:.3f} seconds")

print(f"\n{'=' * 60}")
if time_unpat > time_part:
    speedup = time_unpat / time_part
    print(f"Speedup: {speedup:.2f}x faster with partitioning")
    print(f"Partitioned table skipped ~4/5 of the data (~122 days out of ~153)")
else:
    print(f"Times are similar (dataset size and local mode limits speedup)")
    print(f"However, partitioned table still read less data from storage")
print(f"{'=' * 60}")

print("\nIn production with larger datasets:")
print("   - Speedups of 10-100x are common with good partitioning")
print("   - The benefit scales with data size and query selectivity")
print("   - Partition pruning reduces I/O significantly")
print("   - With 5 months of data, querying 1 month means 4/5 less I/O!")

## File-Level Analysis

Let's look at how data is distributed across files for different queries.

In [ ]:
# Analyze which files would be read for a specific query
print("Files that would be read for August 2023 query:")
spark.sql("""
    SELECT 
        file_path,
        partition,
        record_count,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb
    FROM polaris.taxi.trips_by_day.files
    WHERE partition['tpep_pickup_datetime_day'] >= 19205  -- 2023-08-01
      AND partition['tpep_pickup_datetime_day'] < 19236  -- 2023-09-01
    ORDER BY partition
    LIMIT 10
""").show(truncate=False)

print("\n(Showing first 10 files out of ~31 files for August)")

In [ ]:
# Calculate I/O savings from partition pruning
total_files = spark.sql("""
    SELECT 
        COUNT(*) as file_count,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_mb
    FROM polaris.taxi.trips_by_day.files
""").collect()[0]

filtered_files = spark.sql("""
    SELECT 
        COUNT(*) as file_count,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_mb
    FROM polaris.taxi.trips_by_day.files
    WHERE partition['tpep_pickup_datetime_day'] >= 19205
      AND partition['tpep_pickup_datetime_day'] < 19236
""").collect()[0]

print("I/O Analysis (August 2023 query vs. full 5 months):")
print("=" * 60)
print(f"Total files in table (5 months): {total_files['file_count']}")
print(f"Total size: {total_files['total_mb']} MB")
print(f"\nFiles to read with partition filter (August only): {filtered_files['file_count']}")
print(f"Size to read: {filtered_files['total_mb']} MB")
print(f"\nFiles skipped: {total_files['file_count'] - filtered_files['file_count']}")
if total_files['total_mb'] > 0:
    pct_saved = (1 - filtered_files['total_mb'] / total_files['total_mb']) * 100
    print(f"I/O saved: {pct_saved:.1f}%")
    print(f"\nPartitioning allowed us to skip ~{pct_saved:.0f}% of the data!")
print("=" * 60)

## Summary and Best Practices

### What We Learned:

1. **Partitioning Strategies**:
   - Unpartitioned: Simple, good for small tables or full scans
   - Time-based (month/day): Perfect for time-series queries
   - Categorical: Good for filtering by specific values
   - Consider data skew when partitioning by cardinality

2. **Sort Orders**:
   - Help with range queries and metric pushdown
   - Cluster related data together
   - Use for columns frequently used in WHERE clauses

3. **Metadata Tables**:
   - `files`: See physical file layout and statistics
   - `partitions`: Understand partition distribution
   - `snapshots`: Track table history

4. **Query Optimization**:
   - Partition pruning: Skip entire partitions
   - Metric pushdown: Skip files using min/max statistics
   - Combine both for maximum efficiency

5. **Spark UI**:
   - Essential for understanding query performance
   - Shows what data was actually read
   - Reveals how Spark mapped files to tasks

### Best Practices:

- **Partition by query patterns**: Choose partitions based on common filters
- **Avoid over-partitioning**: Too many small files hurt performance
- **Monitor partition sizes**: Aim for 100MB - 1GB per partition
- **Use Iceberg's hidden partitioning**: No need to include partition columns in queries
- **Leverage sort orders**: For range queries on non-partition columns
- **Check metadata tables**: Before and after changes to understand impact
- **Use EXPLAIN**: Understand query plans before running on large datasets

### Further Exploration:

- Try different partition strategies for your use cases
- Experiment with multi-column partitioning
- Test partition evolution (changing partition specs)
- Explore table maintenance (compaction, expiring snapshots)
- Compare performance across different engines (Spark vs Trino)